# Screen Macro -- Object Detection Training (YOLO, Segmentation)

Segmentation variant of `train_object_detector_yolo.ipynb`, training `yolo11s-seg.pt` (real instance segmentation) on a polygon-labeled dataset instead of `yolo11s.pt`'s plain box detection -- on the theory that real per-pixel mask supervision gives the model a tighter, more reliable sense of the object's actual shape than a box alone. Worth trying since a first head-to-head YOLO-vs-RF-DETR run on a small (20-image) box-labeled dataset showed a real reliability gap worth investigating (see the main game-bot repo's `docs/cv-object-detection-investigation.md`).

**Exports the exact same plain `[N,6]` box-only ONNX contract as `train_object_detector_yolo.ipynb`** -- no mask/proto output, no app-side changes needed. Segmentation is training-time supervision only. Confirmed directly against Ultralytics' pinned `8.4.137` source that a segmentation model's own `model.export(..., nms=True)` does **not** naturally give a bare `[N,6]` tensor -- it appends 32 mask coefficients per row and adds a second prototype-mask output tensor (`NMSModel.forward`, `ultralytics/engine/exporter.py`), and no export flag exists to suppress that. Step 5 below works around this by reusing Ultralytics' own `NMSModel` directly (the exact class the stock exporter itself uses for `nms=True`) and slicing its output down to the first 6 columns before tracing, rather than reimplementing NMS from scratch or shipping mask data the app has no use for. **This custom export cell is the least-verified part of this notebook** -- confirmed at the source level, not yet run end-to-end against a real trained checkpoint.

**Requires a fully polygon-labeled dataset -- not a "mostly boxes, some polygons" mix.** Confirmed directly against Ultralytics' own dataset loader: a segment-task dataset requires every single labeled object across the whole dataset to have a polygon (`YOLODataset.verify_labels` raises "Segment dataset requires equal numbers of boxes and segments" otherwise) -- there's no partial/graceful fallback the way this notebook's RF-DETR sibling (`train_object_detector_seg.ipynb`) has one. This happens to already be guaranteed by how Screen Macro's Label Frames works (one annotation shape -- Bounding Box or Polygon -- per whole dataset, never mixed), so just make sure the dataset you upload was actually labeled in Polygon mode. Negative (zero-object) frames are unaffected either way.

**No click-point (keypoint) prediction**, same as `train_object_detector_yolo.ipynb` -- `onnx_detector.py` clicks the box center for any keypoint-less model.

**Colab (default, manual):** same steps as the other notebooks -- `Runtime` -> `Change runtime type` -> GPU, `Runtime` -> `Run all`, upload the polygon-labeled `..._dataset.zip` when prompted, `best.onnx` downloads automatically at the end.

**Kaggle (opt-in, scripted):** same as the other two notebooks in this repo.

**Status: not yet run against a real dataset.** Code-complete against Ultralytics' documented segmentation training/export API, confirmed directly against the pinned `8.4.137` source (see each step's own markdown for what was actually verified) -- but expect at least one round of real-Colab-run debugging before treating this as proven, matching this whole notebook family's own history. The custom export approach in step 5 in particular (see above) is the part most likely to need real-run fixes.

## 1. Install dependencies

Same pinned versions as `train_object_detector_yolo.ipynb` -- an unpinned install risks a future upstream release silently changing behavior underneath this notebook.

In [ ]:
!pip install -q "ultralytics==8.4.137" "onnx==1.22.0" "onnxruntime==1.29.0"

### Optional: check what hardware this session got

Purely informational -- doesn't change how training runs either way. `BATCH = -1` in step 4 already uses Ultralytics' own AutoBatch to size itself to whatever GPU you're connected to.

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print('\nYour runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
if ram_gb < 20:
    print('Not using a high-RAM runtime')
else:
    print('You are using a high-RAM runtime!')

## 2. Upload and unpack the dataset

Expects the zip Screen Macro's Label Frames (Polygon mode) produces: an `images/` folder of captured frames plus a `labels.json` of the form
`{"images_dir": "images", "annotation_type": "polygon", "labels": [{"image": "frame_001.png", "objects": [{"box": [x, y, w, h], "keypoint": [x, y], "polygon": [[x1, y1], [x2, y2], ...]}, ...]}, ...]}`
(frames with no entry in `labels` were skipped during labeling; each frame can list more than one object). The `keypoint` field is ignored by this notebook, same as the box-only YOLO notebook.

In [ ]:
import os, glob, zipfile, shutil
from pathlib import Path

RAW_DIR = Path("dataset_raw")
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

# cv_training.py's push already uploaded the dataset zip as this kernel's one attached
# Kaggle Dataset -- mounted read-only under /kaggle/input, no interactive upload prompt
# needed the way Colab's files.upload() requires. Detected by whether a zip is actually
# there, not just os.path.exists("/kaggle/input") -- that directory exists (empty) on
# plain Colab too.
candidates = glob.glob("/kaggle/input/*/*.zip") + glob.glob("/kaggle/input/*.zip")
ON_KAGGLE = bool(candidates)
if ON_KAGGLE:
    zip_name = candidates[0]
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(RAW_DIR)

print("Extracted:", list(RAW_DIR.iterdir()))

## 3. Convert to YOLO segmentation format (train/val split)

One class (`"object"`, id `0`), segment-task label format: `class_id x1 y1 x2 y2 ... xn yn`, all coordinates normalized to [0, 1] by image size (Ultralytics' own segment-mode convention, confirmed against `ultralytics/data/utils.py`'s label verification -- a detect-style 5-number box line mixed into a segment-task label file is a hard error there, not silently tolerated). Every object here must have a `"polygon"` from Screen Macro's Polygon-mode labeling -- see the intro above for why there's no partial-box fallback the way the RF-DETR segmentation notebook has one; this cell raises immediately if the dataset isn't actually polygon-labeled, rather than producing a subtly-broken dataset. A frame with zero labeled objects still gets an empty `.txt` file, a real negative example, same convention as the box-only notebook.

Same 90/10 `VALID_FRACTION` split and small-dataset `val`-falls-back-to-`train` handling as `train_object_detector_yolo.ipynb`.

In [ ]:
import json, random
import cv2

VALID_FRACTION = 0.1
YOLO_DIR = Path("dataset_yolo")
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)

with open(RAW_DIR / "labels.json") as f:
    raw = json.load(f)
images_dir = RAW_DIR / raw.get("images_dir", "images")
entries = raw["labels"]

annotation_type = raw.get("annotation_type", "box")
if annotation_type != "polygon":
    raise ValueError(
        f"This dataset's annotation_type is {annotation_type!r}, not 'polygon' -- "
        "re-label it with Screen Macro's Label Frames -> Polygon mode first. Ultralytics' "
        "own segment-task training requires every labeled object to have a real polygon, "
        "not just some (see this step's markdown)."
    )

random.seed(0)
random.shuffle(entries)
n_valid = max(1, int(len(entries) * VALID_FRACTION)) if len(entries) > 1 else 0
splits = {"val": entries[:n_valid], "train": entries[n_valid:]}
if not splits["val"]:
    print("Dataset too small for a real validation split -- reusing the training images for "
          "'val' instead, same as train_object_detector_yolo.ipynb.")
    splits["val"] = splits["train"]

for split_name, split_entries in splits.items():
    (YOLO_DIR / "images" / split_name).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split_name).mkdir(parents=True, exist_ok=True)
    for e in split_entries:
        src = images_dir / e["image"]
        img = cv2.imread(str(src))
        h, w = img.shape[:2]
        shutil.copy(src, YOLO_DIR / "images" / split_name / e["image"])
        stem = Path(e["image"]).stem
        lines = []
        for obj in e["objects"]:
            polygon = obj.get("polygon")
            if not polygon:
                raise ValueError(
                    f"Object in {e['image']} has no 'polygon' -- this dataset isn't fully "
                    "polygon-labeled (see this step's markdown)."
                )
            coords = " ".join(f"{x / w:.6f} {y / h:.6f}" for x, y in polygon)
            lines.append(f"0 {coords}")
        (YOLO_DIR / "labels" / split_name / f"{stem}.txt").write_text("\n".join(lines))
    print(f"{split_name}: {len(split_entries)} image(s)")

data_yaml = YOLO_DIR / "data.yaml"
data_yaml.write_text(
    f"path: {YOLO_DIR.resolve()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: object\n"
)
print("Wrote", data_yaml)

## 4. Train

`yolo11s-seg.pt` (Ultralytics YOLO11-seg, small) instead of the box-only notebook's `yolo11s.pt` -- same size-tradeoff reasoning as that notebook's own `yolo11s.pt` choice. Confirmed directly against Ultralytics' pinned `8.4.137` source that the box branch (box/cls/dfl loss) is inherited unchanged from plain detection with the mask loss simply added on top (`v8SegmentationLoss(v8DetectionLoss)`, `ultralytics/utils/loss.py`) -- so box quality should only gain signal from the extra mask supervision, never regress from it.

Same conservative augmentation set as the box-only notebook (whole-frame flips/rotation/shear/perspective/mosaic/mixup/copy_paste all off, mild color/translate/scale jitter kept) -- that reasoning is about the augmentation itself, not the model architecture, so it carries over unchanged.

`BATCH = -1` (Ultralytics AutoBatch) -- confirmed no segment-specific risk in `ultralytics/utils/autobatch.py`: it profiles memory by running real forward passes on dummy image tensors at increasing batch sizes, with no per-task synthetic-target-dict construction the way RF-DETR's own prober has (the confirmed cause of that notebook family's keypoint-mode auto-batch bug) -- so there's no equivalent failure mode to work around here.

In [ ]:
from ultralytics import YOLO

EPOCHS = 200      # small dataset -- more epochs than a COCO-scale run, early stopping cuts it short once it plateaus
PATIENCE = 20     # stop once validation mAP hasn't improved for this many epochs
IMGSZ = 640
BATCH = -1        # Ultralytics' own AutoBatch -- sizes itself to whatever GPU is actually connected

model = YOLO("yolo11s-seg.pt")
model.train(
    data=str(YOLO_DIR / "data.yaml"),
    epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    # Whole-frame geometric augmentation deliberately conservative/off, same reasoning as
    # train_object_detector_yolo.ipynb's own training cell.
    fliplr=0.0, flipud=0.0, degrees=0.0, shear=0.0, perspective=0.0,
    mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0,
    hsv_h=0.015, hsv_s=0.3, hsv_v=0.2, translate=0.1, scale=0.2,
)

## 5. Export to ONNX with the app's expected contract

Segmentation export needs a different approach from the box-only notebook's plain `model.export(format="onnx", nms=True, ...)` -- confirmed directly against Ultralytics' pinned `8.4.137` source (`ultralytics/engine/exporter.py`'s `NMSModel.forward`) that `nms=True` on a `yolo11s-seg.pt`-trained model does **not** produce a bare `[N,6]` tensor: each detection row gets 32 mask coefficients appended (`(1, max_det, 38)`), plus a second output tensor for the prototype masks, and no export flag exists to suppress this. Ultralytics itself always puts box/score/class in the exact same first-6-columns layout as plain detection though (`dets = torch.cat([box, score, cls, extra], dim=-1)`), so rather than reimplementing NMS by hand, this reuses Ultralytics' own real `NMSModel` -- the exact class the stock exporter itself instantiates for `nms=True` -- and simply slices its output down to those first 6 columns before tracing. This keeps the real IoU-based NMS logic identical to plain detect export, and never traces the prototype-mask branch as a graph output at all.

`conf=0.001` (not Ultralytics' default `0.25`) -- same reasoning as the box-only notebook: keep every meaningfully-non-zero candidate in the output and let the app's own Detect Model action/Validate decide the real confidence cutoff at runtime.

**Least-verified cell in this notebook** (see the intro's own status note) -- `NMSModel`'s exact constructor requirements were confirmed at the source level (the fields it reads off its `args` namespace), but this exact combination hasn't been run against a real trained checkpoint yet. Also doesn't run `onnxsim` (ONNX-simplifier) the way `model.export(simplify=True)` normally would, since we're bypassing that high-level path entirely -- a purely optional graph-cleanup step, safe to run manually afterward (`pip install onnxsim`, then `onnxsim best.onnx best.onnx`) if you want it, not required for the model to work correctly in the app.

Exports from `model.trainer.best` (the best checkpoint path), same discipline as the box-only notebook. `dynamo=False` on the `torch.onnx.export` call below, same reasoning as both RF-DETR notebooks -- the newer default exporter needs the `onnxscript` package (not installed here) and cannot trace this wrapper anyway. Also replicates the handful of export-mode attributes (`m.export = True`, `m.format`, `m.dynamic`, `m.max_det`, `m.xyxy`, ...) Ultralytics' own `Exporter.__call__` sets on every `Detect`/`Segment` head submodule before ever constructing `NMSModel` -- skipped on the first real run of this cell, which crashed `NMSModel.forward` with `AttributeError: 'tuple' object has no attribute 'device'` (the head's forward stays in its raw training-shaped output without `export = True` set).

In [ ]:
from types import SimpleNamespace

import torch
import torch.nn as nn
from ultralytics.engine.exporter import NMSModel
from ultralytics.nn.modules import Detect

best_pt = model.trainer.best
seg_model = YOLO(best_pt)

# Same fields NMSModel.forward itself reads (ultralytics/engine/exporter.py) -- built by
# hand since we're constructing NMSModel directly instead of going through the full
# Exporter, which is what always wraps mask/proto output around it for segment tasks --
# see this step's markdown for why that default path doesn't give a bare [N,6] tensor.
export_args = SimpleNamespace(
    format="onnx", dynamic=False, batch=1, max_det=300,
    conf=0.001,  # keep weak detections in the output -- see this step's markdown for why
    iou=0.7, agnostic_nms=False, opset=17, quantize=False,
)

# Ultralytics' own Exporter.__call__ does exactly this preprocessing on every Detect/
# Segment head submodule before ever constructing NMSModel (confirmed against the pinned
# 8.4.137 source) -- without it, the head's forward() stays in its raw training-oriented
# shape (a nested tuple of per-scale tensors) instead of one concatenated inference
# tensor, which is exactly what NMSModel.forward's own `pred.device` access needs and
# crashed on (AttributeError: 'tuple' object has no attribute 'device') the first time
# this cell actually ran. Missed originally because constructing NMSModel directly (see
# this step's markdown) skips all of Exporter's own setup that normally runs first.
net = seg_model.model
net.eval()
net.float()
for p in net.parameters():
    p.requires_grad = False
net = net.fuse(imgsz=(IMGSZ, IMGSZ))
for m in net.modules():
    if isinstance(m, Detect):  # Segment is a Detect subclass, same as every other head type
        m.dynamic = export_args.dynamic
        m.export = True
        m.format = export_args.format
        available = sum(int(IMGSZ / s) * int(IMGSZ / s) for s in net.stride.tolist())
        m.max_det = min(export_args.max_det, available)
        m.agnostic_nms = export_args.agnostic_nms
        m.xyxy = True  # always want xyxy boxes here (nms=True, format="onnx")
        m.shape = None


class BoxOnlyExport(nn.Module):
    '''Wraps Ultralytics' own NMSModel (the same class the stock exporter uses for
    nms=True) and drops everything past the first 6 columns -- box, score, class -- so a
    segmentation model's mask coefficients and prototype-mask tensor never reach the
    exported graph at all. See this step's markdown for why nms=True alone isn't enough
    for a segment-task model.'''

    def __init__(self, nms_model: nn.Module):
        super().__init__()
        self.nms_model = nms_model

    def forward(self, x):
        out = self.nms_model(x)
        det = out[0] if isinstance(out, tuple) else out  # (detections, proto) for segment tasks
        return det[..., :6]


nms_model = NMSModel(net, export_args)
nms_model.eval()
wrapper = BoxOnlyExport(nms_model)
wrapper.eval()

dummy_input = torch.zeros(1, 3, IMGSZ, IMGSZ)

with torch.no_grad():
    for _ in range(2):  # same dry-run warmup Exporter.__call__ does before tracing
        wrapper(dummy_input)
    torch.onnx.export(
        wrapper, dummy_input, "best.onnx",
        input_names=["images"], output_names=["output0"],
        opset_version=17,
        dynamo=False,  # the newer exporter needs onnxscript (not installed here) and can't trace this wrapper anyway
    )
print("Exported best.onnx from", best_pt, "(segmentation-trained, box-only output)")

## 6. Sanity-check the export (optional)

Same idea as the other two notebooks' own sanity checks. Uses a plain resize rather than a true letterbox for simplicity -- good enough to catch a badly broken export, not meant to exactly reproduce the app's own preprocessing. Safe to skip if you'd rather just download `best.onnx` and test it directly in the app.

In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image

sess = ort.InferenceSession("best.onnx", providers=["CPUExecutionProvider"])
img = Image.open(next((YOLO_DIR / "images" / "val").glob("*.png"))).convert("RGB").resize((IMGSZ, IMGSZ))
blob = (np.array(img).astype(np.float32) / 255.0).transpose(2, 0, 1)[None, ...]
output = sess.run(None, {sess.get_inputs()[0].name: blob})[0]
if output.ndim == 3:
    output = output[0]
print("output shape:", output.shape)  # (max_det, 6): x1,y1,x2,y2,score,label
top = output[np.argsort(-output[:, 4])[:5]]
print(top)

## 7. Download the trained model

**Slow on Colab?** `files.download()` below doesn't do a normal HTTP download -- it base64-encodes the whole file and pushes it through Colab's Python-kernel-to-browser message bridge, which is known to be slow for anything more than a few MB. If you've already mounted your Google Drive in this session, set `USE_GOOGLE_DRIVE = True` below to copy `best.onnx` there instead and download it from drive.google.com -- skips the slow bridge entirely.

In [ ]:
if ON_KAGGLE:
    # Kaggle captures every file left in /kaggle/working/ as this kernel's output -- no
    # equivalent of Colab's interactive download; cv_training.py's poll/pull step fetches
    # it afterward via `kaggle kernels output`.
    print("On Kaggle: best.onnx left in /kaggle/working/ -- fetched by the app's Kaggle poll/pull step.")
else:
    USE_GOOGLE_DRIVE = False  # set True if you've already mounted Drive this session -- see this cell's markdown

    if USE_GOOGLE_DRIVE:
        drive_dest = Path("/content/drive/MyDrive/best.onnx")
        if not drive_dest.parent.is_dir():
            raise RuntimeError(
                "Google Drive isn't mounted at /content/drive -- run "
                "`from google.colab import drive; drive.mount('/content/drive')` in a separate cell "
                "first (one-time permission prompt), then re-run this cell."
            )
        shutil.copy("best.onnx", drive_dest)
        print(f"Copied to Google Drive: {drive_dest} -- download it from drive.google.com.")
    else:
        from google.colab import files
        files.download("best.onnx")